# Инфраструктура для моделей машинного обучения. Практическая работа

# Цель практической работы

Потренироваться в использовании библиотек PySpark SQL и PySpark ML для предобработки данных и обучения моделей.

# Что входит в практическую работу

1. Инициализация спарк-сессии.
2. Загрузка данных.
3. Ознакомление с данными.
4. Преобразование типов столбцов.
5. Очистка данных.
6. Feature-инжиниринг.
7. Векторизация фичей.
8. Создание и обучение модели.
9. Выбор лучшей модели.
10. Обратная связь.


# Что оценивается

- Пройдены все этапы работы.
- Спарк-сессия успешно запущена.
- Данные прочитаны.
- Все колонки с числовыми значениями преобразованы в числовые типы данных (Int или Double).
- Отфильтрованы все строки с Null-значениями.
- Созданы новые фичи.
- Все категориальные колонки преобразованы в числовой вид, выполнены все этапы векторизации признаков.
- Выборка разделена на обучающую и тестовую.
- Создано три объекта: модель, сетка гиперпараметров и evaluator.
- Создан объект класса CrossValidator и обучен на обучающей выборке.
- Выбрана лучшая модель, посчитана метрика качества лучшей модели.


# Задача

Используя данные о клиентах телекоммуникационной компании, обучите модель, предсказывающую их отток.

Описание данных, с которыми вы будете работать:

* **CustomerID**: ID клиента.
* **Gender**: пол клиента.
* **SeniorCitizen**: пенсионер ли клиент (1 — да, 0 — нет).
* **Partner**: есть у клиента партнёр (жена, муж) или нет (Yes/No).
* **Dependents**: есть ли у клиента инждивенцы, например дети (Yes/No).
* **Tenure**: как много месяцев клиент оставался в компании.
* **PhoneService**: подключена ли у клиента телефонная служба (Yes/No).
* **MultipleLines**: подключено ли несколько телефонных линий (Yes, No, No phone service).
* **InternetService**: интернет-провайдер клиента (DSL, Fiber optic, No).
* **OnlineSecurity**: подключена ли у клиента услуга онлайн-безопасности (Yes, No, No internet service)
* **OnlineBackup**: подключена ли услуга резервного копирования онлайн (Yes, No, No internet service).
* **DeviceProtection**: подключена ли услуга защиты устройства (Yes, No, No internet service)
* **TechSupport**: есть ли у клиента техническая поддержка (Yes, No, No internet service).
* **StreamingTV**: подключена ли услуга потокового телевидения (Yes, No, No internet service).
* **StreamingMovies**: подключена ли услуга стримингового воспроизведения фильмов (Yes, No, No internet service).
* **Contract**: тип контракта клиента (Month-to-month, One year, Two year).
* **PaperlessBilling**: есть ли безбумажный счёт.
* **PaymentMethod**: способ оплаты услуг (Electronic check, Mailed check, Bank transfer (automatic), Credit card (automatic)).
* **MonthlyCharges**: сумма, которая списывается ежемесячно.
* **TotalCharges**: сумма, списанная за всё время.
* **Churn**: ушёл ли клиент (Yes/No). Это целевая переменная, которую нужно предсказать.


# 1. Инициализация спарк-сессии

Инициализируйте спарк-сессию.

Эта ячейка нужна для того, чтобы заргузить необходимые библиотеки и настроить окружение Google Colab для работы со Spark.

Просто запустите её перед выполением задания :)

In [17]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"


In [18]:
from pyspark.sql import SparkSession

### Ваш код здесь ###
spark = SparkSession.builder\
        .master("local[*]")\
        .appName('PySpark_Tutorial')\
        .config("spark.driver.memory", "16g")\
        .getOrCreate()

# 2. Загрузка данных
Загрузите данные, сохраните их в переменную типа sparkDataframe, используя метод read.csv (не забывайте про header и delimiter).

In [19]:
### Ваш код здесь ###
df = spark.read.option("header",True).option("delimiter",",").csv("20.6_data.csv")


# 3. Ознакомление с данными
1. Выведите на экран первые несколько строк датафрейма.


In [20]:
### Ваш код здесь ###
df.show(7)

+----------+------+-------------+-------+----------+------+------------+----------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+--------------------+--------------+------------+-----+
|customerID|gender|SeniorCitizen|Partner|Dependents|tenure|PhoneService|   MultipleLines|InternetService|OnlineSecurity|OnlineBackup|DeviceProtection|TechSupport|StreamingTV|StreamingMovies|      Contract|PaperlessBilling|       PaymentMethod|MonthlyCharges|TotalCharges|Churn|
+----------+------+-------------+-------+----------+------+------------+----------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+--------------------+--------------+------------+-----+
|7590-VHVEG|Female|            0|    Yes|        No|     1|          No|No phone service|            DSL|            No|         Yes|              No|         No|    


2. Выведите общее количество строк датафрейма.



In [21]:
### Ваш код здесь ###
df.count()

7043

3. Выведите структуру (схему) датафрейма.

In [22]:
### Ваш код здесь ###
df.printSchema()

root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: string (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: string (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- MonthlyCharges: string (nullable = true)
 |-- TotalCharges: string (nullable = true)
 |-- Churn: string (nullable = true)



# 4. Преобразование типов столбцов
Преобразуйте тип столбцов у числовых признаков (Int — если признак целочисленный, Double — если признак не целочисленный). Сохраните преобразованный датафрейм в новую переменную.

## Совет

Если вам сложно выполнить это задание, изучите дополнительные материалы: [об операторе Cast](https://sparkbyexamples.com/pyspark/pyspark-cast-column-type/), [об операторе Select](https://sparkbyexamples.com/pyspark/select-columns-from-pyspark-dataframe/).



In [23]:
from pyspark.sql.functions import expr, col

### Ваш код здесь ###

In [24]:
df_formatted = df.select(
    col("customerID").cast("String"),
    col("gender").cast("String"),
    col("SeniorCitizen").cast("Int"),
    col("Partner").cast("String"),
    col("Dependents").cast("String"),
    col("tenure").cast("Int"),
    col("PhoneService").cast("String"),
    col("MultipleLines").cast("String"),
    col("InternetService").cast("String"),
    col("OnlineSecurity").cast("String"),
    col("OnlineBackup").cast("String"),
    col("DeviceProtection").cast("String"),
    col("TechSupport").cast("String"),
    col("StreamingTV").cast("String"),
    col("StreamingMovies").cast("String"),
    col("Contract").cast("String"),
    col("PaperlessBilling").cast("String"),
    col("PaymentMethod").cast("String"),
    col("MonthlyCharges").cast("Double"),
    col("TotalCharges").cast("Double"),
    col("Churn").cast("String")
)

In [25]:
df_formatted.printSchema()

root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: integer (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: double (nullable = true)
 |-- Churn: string (nullable = true)



# 5. Очистка данных
Проверьте, есть ли в какой-либо колонке Null-значения. Для этого можно использовать your_dataframe.filter(col("colname")).isNull()).

Выведите на экран несколько строк с Null-значениями в одной из колонок.

Сохраните очищенный от строк с Null-значениями датафрейм в новую переменную. Для фильтрации этих значений можно использовать метод .isNotNull().

Колонок в датафрейме много, проверять каждую неудобно и долго. Подумайте, как упроситить эту работу, если использовать, например, перебор с циклом for.

[Примеры использования операторов isNull() и isNotNull()](https://sparkbyexamples.com/pyspark/pyspark-isnull/).


In [26]:
### Ваш код здесь ###
df_formatted.filter(col('TotalCharges').isNull()).show()
# Получение списка имен всех колонок в датафрейме
columns = df_formatted.columns

# Перебор всех колонок и проверка Null значений
for column in columns:
    if df_formatted.filter(col(column).isNull()).count() > 0:
        print(f"Колонка {column} содержит Null-значения")

# Фильтрация строк с Null-значениями
df_notnull = df_formatted.dropna()
df_notnull.show()

+----------+------+-------------+-------+----------+------+------------+----------------+---------------+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+--------+----------------+--------------------+--------------+------------+-----+
|customerID|gender|SeniorCitizen|Partner|Dependents|tenure|PhoneService|   MultipleLines|InternetService|     OnlineSecurity|       OnlineBackup|   DeviceProtection|        TechSupport|        StreamingTV|    StreamingMovies|Contract|PaperlessBilling|       PaymentMethod|MonthlyCharges|TotalCharges|Churn|
+----------+------+-------------+-------+----------+------+------------+----------------+---------------+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+--------+----------------+--------------------+--------------+------------+-----+
|4472-LVYGI|Female|            0|    Yes|       Yes|     0|          No|No phon

# 6. Feature-инжиниринг
Добавьте в датафрейм одну или несколько новых фичей. Удалите колонки, которые, как вам кажется, нужно убрать из фичей. Обоснуйте свои решения.

In [27]:
from pyspark.sql.functions import when

### Ваш код здесь ###

#CustomerID: уникальный идентификатор клиента. Этот атрибут не содержит полезной информации для моделирования и может быть удален из датафрейма.
#Удаляем фичи "PhoneService", "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup", 
    #"DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies". Вместо этих признаков генерируем
    #Количество активных услуг (ActiveServicesCount): это будет новый признак, основанный на столбцах 
    #PhoneService, MultipleLines, InternetService, OnlineSecurity,
    #OnlineBackup, DeviceProtection, TechSupport, StreamingTV и StreamingMovies.

#Средняя стоимость активных услуг (AverageServiceCost): это будет новый признак, основанный на столбцах MonthlyCharges и ActiveServicesCount.

#Средний платеж в месяц (AverageMonthlyPayment): это будет новый признак, основанный на столбцах TotalCharges и tenure

df_new_feats = df_notnull.withColumn('ActiveServicesCount',
                                             when(df_notnull['PhoneService'] == 'Yes', 1).otherwise(0) +
                                             when(df_notnull['MultipleLines'] == 'Yes', 1).otherwise(0) +
                                             when(df_notnull['InternetService'] != 'No', 1).otherwise(0) +
                                             when(df_notnull['OnlineSecurity'] == 'Yes', 1).otherwise(0) +
                                             when(df_notnull['OnlineBackup']  == 'Yes', 1).otherwise(0) +
                                             when(df_notnull['DeviceProtection']  == 'Yes', 1).otherwise(0) +
                                             when(df_notnull['TechSupport']  == 'Yes', 1).otherwise(0) +
                                             when(df_notnull['StreamingTV']  == 'Yes', 1).otherwise(0) +
                                             when(df_notnull['StreamingMovies']  == 'Yes', 1).otherwise(0)).\
                                   withColumn('AverageServiceCost', col('MonthlyCharges') / col('ActiveServicesCount')).\
                                   withColumn('AverageMonthlyPayment', col('TotalCharges') / col('tenure')).\
                                   drop('CustomerID', "PhoneService", "MultipleLines", "InternetService", 
                                        "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", 
                                        "StreamingTV", "StreamingMovies")
df_new_feats.show()
df_new_feats.printSchema()

+------+-------------+-------+----------+------+--------------+----------------+--------------------+--------------+------------+-----+-------------------+------------------+---------------------+
|gender|SeniorCitizen|Partner|Dependents|tenure|      Contract|PaperlessBilling|       PaymentMethod|MonthlyCharges|TotalCharges|Churn|ActiveServicesCount|AverageServiceCost|AverageMonthlyPayment|
+------+-------------+-------+----------+------+--------------+----------------+--------------------+--------------+------------+-----+-------------------+------------------+---------------------+
|Female|            0|    Yes|        No|     1|Month-to-month|             Yes|    Electronic check|         29.85|       29.85|   No|                  2|            14.925|                29.85|
|  Male|            0|     No|        No|    34|      One year|              No|        Mailed check|         56.95|      1889.5|   No|                  4|           14.2375|     55.5735294117647|
|  Male|       

#7. Векторизация фичей
Подготовьте данные к обучению:





1. Преобразуйте текстовые колонки в числа, используя StringIndexer.
Удалите столбцы со старыми (непреобразованными) признаками. Выведите на экран структуру получившегося датафрейма. Не забывайте о столбце Churn. Хоть он и выступает в задаче как таргет, он имеет текстовый тип, поэтому тоже должен быть закодирован числовыми значениями.

Чтобы использовать StringIndexer для всех категориальных признаков сразу, а не для каждого отдельно, можно применить сущность pipeline.

**Пример кода:**

##### #Задаём список текстовых колонок:
text_columns = ["text_col_1", "text_col_2", "text_col_3"]

##### #Задаём список StringIndexer'ов — сущностей, каждая из которых будет кодировать одну текстовую колонку числами. Имена преобразованных колонок будут заканчиваться на _index:
indexers = [StringIndexer(inputCol=column, outputCol=column+"_index",).fit(<ваш датасет>) for column in text_columns]

##### #Создаём Pipeline из StringIndexer'ов:
pipeline = Pipeline(stages=indexers)

##### #Скармливаем нашему pipeline датафрейм, удаляя старые колонки:
new_dataframe = pipeline.fit(<ваш датасет>).transform(<ваш датасет>).drop(*text_columns)


In [28]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline
spark.conf.set("spark.sql.debug.maxToStringFields", "50")
# Список колонок с текстовым типом
text_cols = ["gender", "Partner", "Dependents", 
             "Contract", "PaperlessBilling", "PaymentMethod", "Churn"]

### Ваш код здесь ###
# Задаем список StringIndexer'ов
indexers = [StringIndexer(inputCol=column, outputCol=column+"_index").fit(df)
            for column in text_cols]

# Создаем Pipeline из StringIndexer'ов
pipeline = Pipeline(stages=indexers)

# Преобразуем данные с помощью Pipeline и удаляем старые колонки
df_transformed = pipeline.fit(df_new_feats).transform(df_new_feats).drop(*text_cols)

2. Векторизуйте категориальные признаки, используя OneHotEncoder.
Удалите столбцы со старыми (непреобразованными) признаками.
Выведите на экран структуру получившегося после преобразований датафрейма.


In [29]:
# Векторизуем категориальные признаки с помощью OneHotEncoder и удаляем старые колонки
text_cols = ["gender", "Partner", "Dependents", 
             "Contract", "PaperlessBilling", "PaymentMethod"]

### Ваш код здесь ###
encoder = OneHotEncoder(inputCols=[column+"_index" for column in text_cols],
                        outputCols=[column+"_encoded" for column in text_cols])
df_encoded = encoder.fit(df_transformed).transform(df_transformed).withColumnRenamed('Churn_index', 'Churn').\
                     drop(*[column+"_index" for column in text_cols])
df_encoded.printSchema()

root
 |-- SeniorCitizen: integer (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: double (nullable = true)
 |-- ActiveServicesCount: integer (nullable = false)
 |-- AverageServiceCost: double (nullable = true)
 |-- AverageMonthlyPayment: double (nullable = true)
 |-- Churn: double (nullable = false)
 |-- gender_encoded: vector (nullable = true)
 |-- Partner_encoded: vector (nullable = true)
 |-- Dependents_encoded: vector (nullable = true)
 |-- Contract_encoded: vector (nullable = true)
 |-- PaperlessBilling_encoded: vector (nullable = true)
 |-- PaymentMethod_encoded: vector (nullable = true)



3. Объедините колонки фичей в один вектор, используя VectorAssembler.
Удалите столбцы со старыми (непреобразованными) признаками.
Выведите на экран первые несколько строк и структуру получившегося датафрейма.

In [30]:
### Ваш код здесь ###
# Объединяем колонки фичей в один вектор с помощью VectorAssembler
features = list(df_encoded.drop('Churn').columns) #Список колонок фичей
            
assembler = VectorAssembler(inputCols=features, outputCol="features")
df_assembled = assembler.transform(df_encoded).select("features", 'Churn')

# Выводим первые несколько строк и структуру получившегося датафрейма

df_assembled.show()
df_assembled.printSchema()

+--------------------+-----+
|            features|Churn|
+--------------------+-----+
|[0.0,1.0,29.85,29...|  0.0|
|[0.0,34.0,56.95,1...|  0.0|
|[0.0,2.0,53.85,10...|  1.0|
|[0.0,45.0,42.3,18...|  0.0|
|[0.0,2.0,70.7,151...|  1.0|
|[0.0,8.0,99.65,82...|  1.0|
|[0.0,22.0,89.1,19...|  0.0|
|[0.0,10.0,29.75,3...|  0.0|
|[0.0,28.0,104.8,3...|  1.0|
|(16,[1,2,3,4,5,6,...|  0.0|
|[0.0,13.0,49.95,5...|  0.0|
|[0.0,16.0,18.95,3...|  0.0|
|(16,[1,2,3,4,5,6,...|  0.0|
|[0.0,49.0,103.7,5...|  1.0|
|[0.0,25.0,105.5,2...|  0.0|
|(16,[1,2,3,4,5,6,...|  0.0|
|(16,[1,2,3,4,5,6,...|  0.0|
|[0.0,71.0,106.7,7...|  0.0|
|(16,[1,2,3,4,5,6,...|  1.0|
|[0.0,21.0,90.05,1...|  0.0|
+--------------------+-----+
only showing top 20 rows

root
 |-- features: vector (nullable = true)
 |-- Churn: double (nullable = false)



#8. Создание и обучение модели

1. Создайте модель — логистическую регрессию (используя LogisticRegression). В качестве параметров класса LogisticRegression укажите колонку фичей (параметр featuresCol), колонку-таргет (параметр labelCol) из датафрейма и имя колонки, в которую будут записываться предсказания (параметр predictionCol).

In [31]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import BinaryClassificationEvaluator

### Ваш код здесь ###
# Создание модели логистической регрессии
lr = LogisticRegression(featuresCol="features", labelCol="Churn", predictionCol="prediction")

2. Разделите датафрейм на обучающую и тестовую выборку.

In [32]:
### Ваш код здесь ###
# Разделение датафрейма на обучающую и тестовую выборки
train, test = df_assembled.randomSplit([0.7, 0.3], seed=42)

3. Создайте объекты — сетки гиперпараметров для каждой модели, используя ParamGridBuilder. Так же, как и в ноутбуке из последнего видео, в сетку гиперпараметров можно добавить значения параметров regParam и elasticNetParam.

Вы можете ознакомиться [с документацией объекта LogisticRegression в PySpark](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.classification.LogisticRegression.html) и добавить в сетку больше параметров.


In [33]:
### Ваш код здесь ###
# Создание сетки гиперпараметров
paramGrid = ParamGridBuilder().addGrid(lr.regParam, [0.01, 0.05, 0.1]).\
                               addGrid(lr.elasticNetParam, [0.01, 0.1]).\
                               addGrid(lr.threshold, [0.3, 0.5, 0.7]).\
                               build()

4. Создайте объект evaluator, который будет отвечать за метрику качества при обучении. Для этого используйте класс BinaryClassificationEvaluator со следующими параметрами: rawPredictionCol — колонка с предсказаниями, labelCol — колонка с таргетом.

У вас, возможно, возник вопрос, какую метрику качества берёт по умолчанию BinaryClassificationEvaluator. По умолчанию BinaryClassificationEvaluator будет рассчитывать areaUnderROC. Это метрика оценки площади под кривой ROC (Receiver Operating Characteristic), которая служит графической интерпретацией производительности модели. Эта метрика качества находится в пределах от 0 до 1. Чем выше метрика, тем более качественные предсказания делает модель.

In [34]:
### Ваш код здесь ###
# Создание evaluator
evaluator = BinaryClassificationEvaluator(rawPredictionCol="prediction", labelCol="Churn")

5. Создайте объект CrossValidator, в качестве параметров укажите уже созданные вами модель, сетку гиперпараметров и evaluator.

In [35]:
### Ваш код здесь ###
# Создание CrossValidator
cv = CrossValidator(estimator=lr, estimatorParamMaps=paramGrid, evaluator=evaluator)

6. Запустите обучение модели на тренировочной выборке. Сохраните обученную модель в новую переменную.

In [36]:
### Ваш код здесь ###
# Обучение модели
cvModel = cv.fit(train)

#9. Выбор лучшей модели

1. Выберите лучшую модель, сохраните её в отдельную переменную, отобразите её параметры.

Вывод параметров модели в PySpark можно сделать, используя метод extractParamMap().

In [38]:

### Ваш код здесь ###
# Выбор лучшей модели
bestModel = cvModel.bestModel

# Вывод параметров лучшей модели
print(bestModel.extractParamMap())

{Param(parent='LogisticRegression_5b7f83f9f3d6', name='aggregationDepth', doc='suggested depth for treeAggregate (>= 2).'): 2, Param(parent='LogisticRegression_5b7f83f9f3d6', name='elasticNetParam', doc='the ElasticNet mixing parameter, in range [0, 1]. For alpha = 0, the penalty is an L2 penalty. For alpha = 1, it is an L1 penalty.'): 0.01, Param(parent='LogisticRegression_5b7f83f9f3d6', name='family', doc='The name of family which is a description of the label distribution to be used in the model. Supported options: auto, binomial, multinomial'): 'auto', Param(parent='LogisticRegression_5b7f83f9f3d6', name='featuresCol', doc='features column name.'): 'features', Param(parent='LogisticRegression_5b7f83f9f3d6', name='fitIntercept', doc='whether to fit an intercept term.'): True, Param(parent='LogisticRegression_5b7f83f9f3d6', name='labelCol', doc='label column name.'): 'Churn', Param(parent='LogisticRegression_5b7f83f9f3d6', name='maxBlockSizeInMB', doc='maximum memory in MB for stacki

2. Запустите лучшую модель в режиме предсказания на тестовой выборке. Сохраните предсказания в отдельную переменную. Выведите первые несколько строк датафрейма с предсказаниями на экран.

Запуск модели в режиме предсказания выполняется при помощи метода .transform(<тестовая выборка>).

In [39]:
### Ваш код здесь ###
# Предсказание на тестовой выборке
predictions = bestModel.transform(test)

# Вывод первых нескольких строк датафрейма с предсказаниями
predictions.show()

+--------------------+-----+--------------------+--------------------+----------+
|            features|Churn|       rawPrediction|         probability|prediction|
+--------------------+-----+--------------------+--------------------+----------+
|(16,[0,1,2,3,4,5,...|  0.0|[2.21603872773210...|[0.90168057651507...|       0.0|
|(16,[0,1,2,3,4,5,...|  0.0|[2.56672736632614...|[0.92868926790575...|       0.0|
|(16,[0,1,2,3,4,5,...|  0.0|[4.65111397642600...|[0.99053940130434...|       0.0|
|(16,[0,1,2,3,4,5,...|  0.0|[2.12460236749421...|[0.89327150268305...|       0.0|
|(16,[0,1,2,3,4,5,...|  1.0|[1.67984775144300...|[0.84288436984364...|       0.0|
|(16,[0,1,2,3,4,5,...|  0.0|[1.66217446722437...|[0.84052968349708...|       0.0|
|(16,[0,1,2,3,4,5,...|  0.0|[2.30896816232753...|[0.90961705978171...|       0.0|
|(16,[0,1,2,3,4,5,...|  1.0|[1.81111950929774...|[0.85949712266281...|       0.0|
|(16,[0,1,2,3,4,5,...|  0.0|[4.04677080798110...|[0.98282153196394...|       0.0|
|(16,[0,1,2,3,4,

3. Получите метрику качества модели. Для этого примените к объекту evaluator метод .evaluate(<ваш датафрейм с предсказаниями>).



In [40]:
### Ваш код здесь ###
# Вычисление метрики качества
evaluator.evaluate(predictions)

0.7513152142781773

#10. Обратная связь
Вы ознакомились с возможностями двух мощных библиотек: PySpark SQL для предобработки данных и PySpark ML для машинного обучения.

Поделитесь впечатлениями от работы с новыми библиотеками. В чём они более удобны, чем уже знакомые вам Pandas и Sklearn, а в чём нет.

PySpark SQL и PySpark ML являются более удобными инструментами для обработки и анализа больших данных, чем Pandas и Scikit-Learn (Sklearn) по следующим причинам:

Параллельная обработка: PySpark может обрабатывать и анализировать большие наборы данных параллельно на нескольких узлах кластера Hadoop или Spark. Это значительно ускоряет процесс обработки данных по сравнению с Pandas, который обрабатывает данные последовательно.
Масштабируемость: PySpark способен обрабатывать огромные объемы данных, которые превышают возможности оперативной памяти одного компьютера. Это делает его идеальным инструментом для обработки больших данных.
Поддержка различных форматов данных: PySpark поддерживает различные форматы хранения данных, такие как CSV, JSON, Parquet и Avro, что позволяет работать с данными, полученными из разных источников.
Однако, PySpark имеет и некоторые недостатки по сравнению с Pandas:

Сложность: PySpark является более сложным инструментом по сравнению с Pandas. Он требует более глубокого понимания Spark и его архитектуры, а также настройки кластера для выполнения вычислений.

# Как отправить работу на проверку

Загрузите файл с заданиями, откройте его через Jupyter Notebook в Google Colab. Скачайте файл с датасетом и загрузите его в Colab. Выполните задачи, сохраните изменения: воспользуйтесь опцией Save and Checkpoint из вкладки меню File или кнопкой Save and Checkpoint на панели инструментов. Отправьте через форму ниже итоговый файл Jupyter Notebook (.ipynb) или ссылку на него.